## Influence-driven data curation
Idea: the kernel of the surrogate model represents the relationship between outcomes across the factor space -> it helps us understand how the mean outcome across the factor space could change if one or more outcomes were improved (i.e. collected demos for)

Terms:
- $y_{target}$: the maximum outcome achievable
- $\mu(x)$: expected outcome at factor configuration $x$

For a single candidate demo $x^*$, the expected new mean outcome at any other point $x$ in the space is:
$$\mu_{new}(x) = \mu_{current}(x) + \frac{k(x, x^*)}{k(x^*, x^*) + \sigma^2_{noise}} (y_{target} - \mu_{current}(x^*))$$

To calculate the expected change in mean outcome across the entire factor space, we integrate that change over the whole domain $\mathcal{X}$.
$$\text{Total Influence}(x^*) = \left( y_{target} - \mu_{current}(x^*) \right) \cdot \int_{\mathcal{X}} \frac{k(x, x^*)}{k(x^*, x^*) + \sigma^2_{noise}} dx$$

To find the $x^*$ that maximizes the overall average change, simply calculate this score for all candidates and select the highest one.

What if you wanted to select more than one point (a batch $X_{batch} = [x_1, x_2, \dots, x_N]$) to collect demos for?
To score a full batch, you upgrade the scalar covariance $k(x, x^*)$ into a covariance matrix $K(X_{batch}, X_{batch})$. The expected total influence for a batch becomes:$$\text{Total Influence}(X_{batch}) = \int_{\mathcal{X}} K(x, X_{batch}) \left[ K(X_{batch}, X_{batch}) + \Sigma_{noise} \right]^{-1} (Y_{target} - M_{current}(X_{batch})) dx$$

In [1]:
import numpy as np
import torch
import pickle
import pandas as pd
from factors_config import FACTOR_COLUMNS, get_design_points_robot, BOUNDS, tkwargs

In [2]:
TASKS = ['pickblueblock', 'uprightcup', 'putgreeninpot']
MAX_OUTCOME = {'pickblueblock': 2.0, 'uprightcup': 3.0, 'putgreeninpot': 6.0}

In [3]:
# Load active model (currently only works for SingleTaskGP and FullyBayesianSingleTaskGP)
active_model_path = './results/putgreeninpot_active_offline_FullyBayesianSingleTaskGP_qBALD/run_1/models/trial_100_model.pkl'

with open(active_model_path, 'rb') as f:
    active_model_data = pickle.load(f)
    active_model = active_model_data['model']
    active_model_name = active_model_data.get('model_name', 'Unknown')
print(f"Loaded active model: {active_model_name}")

/home/liao0241/active_testing/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded active model: FullyBayesianSingleTaskGP


In [24]:
# See covariance matrix on training data
bf_data = pd.read_csv('./results/putgreeninpot_bruteforce/results.csv')
print("Full factor space size: ", len(bf_data))

# Optional: Filter factor values i.e. restrict which rows of X_all greedy/local search may pick for the batch.
# Combine these factor value filters as needed
bottom_left = (bf_data['x'].values <= 0.5) & (bf_data['y'].values <= 0.5)
top_left = (bf_data['x'].values <= 0.5) & (bf_data['y'].values >= 0.5)
cam_back = (bf_data['camera_azimuth'].values == 90)
cam_right = (bf_data['camera_azimuth'].values == -30)
th1 = (bf_data['table_height'].values == 1)
th2 = (bf_data['table_height'].values == 2)
th3 = (bf_data['table_height'].values == 3)

candidate_indices = np.where((cam_back | cam_right) & (bottom_left | top_left) & (th1 | th3))[0].tolist()
print("Filtered factor space size: ", len(candidate_indices))

final_bf_data = bf_data.iloc[candidate_indices].reset_index(drop=True)
X_all = torch.tensor(final_bf_data[FACTOR_COLUMNS].values, **tkwargs)
y_all = torch.tensor(final_bf_data['continuous_outcome'].values, **tkwargs).unsqueeze(-1)

# active_model.covar_module(X_all).to_dense()

Full factor space size:  723
Filtered factor space size:  218


### Calculating batch total influence

In [25]:
def calculate_batch_total_influence(X_batch, active_model, X_all, y_target):
    """
    Scores a proposed batch of points based on expected total influence.

    X_all: full discrete grid over which the influence contribution is summed
    (the domain in the integral / sum in the markdown above). X_batch may be any subset
    of rows from this grid or elsewhere in factor space; batch entries must be valid inputs
    to the GP kernel.

    SingleTaskGP: standard (B, B) kernel and scalar / length-1 noise.
    FullyBayesianSingleTaskGP (and similar): kernel and noise are batched over MCMC
    samples (S, B, B) and (S, 1); we average the total-influence scalar over samples.
    """
    active_model.eval()
    active_model.likelihood.eval()

    with torch.no_grad():
        mean_batch = active_model.posterior(X_batch).mean
        K_bb = active_model.covar_module(X_batch).to_dense()
        K_ab = active_model.covar_module(X_all, X_batch).to_dense()
        noise_t = active_model.likelihood.noise

        if K_bb.ndim == 2:
            # (B, B), noise typically shape (1,) for homoskedastic SingleTaskGP
            noise = noise_t.to(dtype=K_bb.dtype, device=K_bb.device).mean().item()
            B = K_bb.shape[0]
            eye_B = torch.eye(B, device=K_bb.device, dtype=K_bb.dtype)
            K_bb_noisy = K_bb + noise * eye_B
            y_diff = y_target - mean_batch
            inv_term = torch.linalg.solve(K_bb_noisy, y_diff)
            influence_per_point = torch.matmul(K_ab, inv_term)
            total_influence = influence_per_point.sum()
        else:
            # (S, B, B) e.g. FullyBayesianSingleTaskGP with S MCMC hyperparameter samples
            S, B, _ = K_bb.shape
            noise = noise_t.squeeze(-1).to(dtype=K_bb.dtype, device=K_bb.device)
            eye_B = torch.eye(B, device=K_bb.device, dtype=K_bb.dtype)
            K_bb_noisy = K_bb + noise.view(S, 1, 1) * eye_B
            y_diff = y_target - mean_batch
            inv_term = torch.linalg.solve(K_bb_noisy, y_diff)
            influence_per_point = torch.matmul(K_ab, inv_term)
            total_influence = influence_per_point.sum(dim=1).mean()

    return total_influence.item()

### Greedy and stochastic greedy batch search

In [26]:
# Greedy batch selection
def propose_batch_greedy(batch_size, active_model, X_all, y_target):
    """
    Greedily builds a batch of size `batch_size` that maximizes total influence.

    X_all: full discrete grid; total influence is summed over all rows (same tensor passed to
    calculate_batch_total_influence as X_all).
    """
    n = X_all.shape[0]
    pool = list(range(n))
    for idx in pool:
        if not (0 <= idx < n):
            raise ValueError(f"candidate index {idx} out of range for X_all with {n} rows.")
    if batch_size > len(pool):
        raise ValueError(
            f"batch_size ({batch_size}) cannot exceed number of candidate indices ({len(pool)})."
        )

    selected_indices = []
    selected_set = set()

    print(f"Proposing a batch of {batch_size} points (candidate pool size {len(pool)})...")

    for step in range(batch_size):
        best_score = -float('inf')
        best_idx = -1

        for i in pool:
            if i in selected_set:
                continue

            current_batch_indices = selected_indices + [i]
            X_batch_candidate = X_all[current_batch_indices]

            score = calculate_batch_total_influence(X_batch_candidate, active_model, X_all, y_target)

            if score > best_score:
                best_score = score
                best_idx = i

        selected_indices.append(best_idx)
        selected_set.add(best_idx)
        print(f"  Step {step+1}/{batch_size}: Added point index {best_idx} | New Batch Influence Score: {best_score:.4f}")

    X_proposed = X_all[selected_indices]
    return X_proposed, selected_indices, best_score

In [27]:
# Stochastic greedy batch selection
import random

def propose_batch_local_search(batch_size, active_model, X_all, y_target, max_iters=1000):
    """
    Improves a batch using Stochastic Local Search (Swapping) over a discrete grid.
    """
    n = X_all.shape[0]
    pool = list(range(n))
    all_indices = set(pool)

    # 1. Initialize with the Greedy approach for a strong head start
    print("Initializing with Greedy search...")
    _, current_indices, best_score = propose_batch_greedy(
        batch_size, active_model, X_all, y_target
    )

    print(f"\nStarting Local Search Optimization. Initial Score: {best_score:.4f}")

    # 2. Iteratively attempt to swap points to find synergistic combinations
    improvements = 0

    for iteration in range(max_iters):
        idx_to_remove = random.choice(current_indices)

        available_indices = list(all_indices - set(current_indices))
        if not available_indices:
            break
        idx_to_add = random.choice(available_indices)

        proposed_indices = current_indices.copy()
        proposed_indices.remove(idx_to_remove)
        proposed_indices.append(idx_to_add)

        proposed_X_batch = X_all[proposed_indices]
        proposed_score = calculate_batch_total_influence(proposed_X_batch, active_model, X_all, y_target)

        if proposed_score > best_score:
            current_indices = proposed_indices
            best_score = proposed_score
            improvements += 1
            print(f"  [Iter {iteration}] Improved! Swapped {idx_to_remove} for {idx_to_add}. New Score: {best_score:.4f}")

    print(f"\nLocal Search Complete. Made {improvements} improvements.")

    final_X_batch = X_all[current_indices]
    return final_X_batch, current_indices, best_score

### Test these selection strategies out

In [31]:
# Define your target/max outcome (e.g. 3.0 for the 'uprightcup' task)
Y_TARGET = 6.0  # change for different tasks
BATCH_SIZE = 24  # how many points to collect demos at (e.g. for fairness when comparing to 'observed failures' method, set to the same number)

In [32]:
# --- Greedy selection ---
greedy_X, greedy_indices, greedy_score = propose_batch_greedy(
    batch_size=BATCH_SIZE,
    active_model=active_model,
    X_all=X_all,
    y_target=Y_TARGET
)

print(f"Greedy Score: {greedy_score:.4f}")

Proposing a batch of 24 points (candidate pool size 218)...
  Step 1/24: Added point index 27 | New Batch Influence Score: 187.0783
  Step 2/24: Added point index 130 | New Batch Influence Score: 299.9117
  Step 3/24: Added point index 85 | New Batch Influence Score: 394.8317
  Step 4/24: Added point index 194 | New Batch Influence Score: 482.0649
  Step 5/24: Added point index 25 | New Batch Influence Score: 515.4819
  Step 6/24: Added point index 161 | New Batch Influence Score: 536.1776
  Step 7/24: Added point index 95 | New Batch Influence Score: 553.9845
  Step 8/24: Added point index 217 | New Batch Influence Score: 570.5925
  Step 9/24: Added point index 47 | New Batch Influence Score: 586.9739
  Step 10/24: Added point index 0 | New Batch Influence Score: 598.3129
  Step 11/24: Added point index 119 | New Batch Influence Score: 607.7074
  Step 12/24: Added point index 172 | New Batch Influence Score: 616.1391
  Step 13/24: Added point index 63 | New Batch Influence Score: 623.

In [38]:
greedy_df = pd.DataFrame(data=greedy_X.numpy(), columns=FACTOR_COLUMNS)
# Show proposed points in a table
# print("\nProposed Factor Configurations to Collect Next:")
# print(greedy_df)

# Save proposed points to csv
greedy_df.to_csv('./results/next_demos/putgreeninpot/next_demos_influence_greedy.csv', index=False)

In [34]:
# --- Stochastic greedy selection ---
stoch_greedy_X, stoch_greedy_indices, stoch_greedy_score = propose_batch_local_search(
    batch_size=BATCH_SIZE,
    active_model=active_model,
    X_all=X_all,
    y_target=Y_TARGET,
    max_iters=1000,  # Adjust based on your compute budget
)

print(f"Stochastic Greedy Score: {stoch_greedy_score:.4f}")

Initializing with Greedy search...
Proposing a batch of 24 points (candidate pool size 218)...
  Step 1/24: Added point index 27 | New Batch Influence Score: 187.0783
  Step 2/24: Added point index 130 | New Batch Influence Score: 299.9117
  Step 3/24: Added point index 85 | New Batch Influence Score: 394.8317
  Step 4/24: Added point index 194 | New Batch Influence Score: 482.0649
  Step 5/24: Added point index 25 | New Batch Influence Score: 515.4819
  Step 6/24: Added point index 161 | New Batch Influence Score: 536.1776
  Step 7/24: Added point index 95 | New Batch Influence Score: 553.9845
  Step 8/24: Added point index 217 | New Batch Influence Score: 570.5925
  Step 9/24: Added point index 47 | New Batch Influence Score: 586.9739
  Step 10/24: Added point index 0 | New Batch Influence Score: 598.3129
  Step 11/24: Added point index 119 | New Batch Influence Score: 607.7074
  Step 12/24: Added point index 172 | New Batch Influence Score: 616.1391
  Step 13/24: Added point index 6

In [39]:
stoch_greedy_df = pd.DataFrame(data=stoch_greedy_X.numpy(), columns=FACTOR_COLUMNS)
# print("\nProposed Factor Configurations to Collect Next:")
# print(stoch_greedy_df)

stoch_greedy_df.to_csv('./results/next_demos/putgreeninpot/next_demos_influence_stoch_greedy.csv', index=False)

In [40]:
# Compare to "certain failures" selection
certain_failures_points = pd.read_csv('./results/next_demos/putgreeninpot/next_demos_certain_failures.csv')
print(f"Loaded {len(certain_failures_points)} points")
cf_X = torch.tensor(certain_failures_points[FACTOR_COLUMNS].values, **tkwargs)
cf_score = calculate_batch_total_influence(cf_X, active_model, X_all, Y_TARGET)
print(f"Certain Failures Score: {cf_score:.4f}")

Loaded 24 points
Certain Failures Score: 500.3921


In [41]:
# Compare to "observed failures" selection
observed_failures_points = pd.read_csv('./results/next_demos/putgreeninpot/next_demos_observed_failures.csv')
print(f"Loaded {len(observed_failures_points)} points")
of_X = torch.tensor(observed_failures_points[FACTOR_COLUMNS].values, **tkwargs)
of_score = calculate_batch_total_influence(of_X, active_model, X_all, Y_TARGET)
print(f"Observed Failures Score: {of_score:.4f}")

Loaded 24 points
Observed Failures Score: 594.7595


In [ ]:
# Compare to random selection
random_scores = []
pool_idx = torch.arange(len(X_all))
if BATCH_SIZE > len(pool_idx):
    raise ValueError("BATCH_SIZE cannot exceed len(candidate pool).")

for i in range(10):
    perm = torch.randperm(len(pool_idx))[:BATCH_SIZE]
    random_indices = pool_idx[perm]
    random_X_batch = X_all[random_indices]
    random_score = calculate_batch_total_influence(random_X_batch, active_model, X_all, Y_TARGET)
    random_scores.append(random_score)

random_score = sum(random_scores) / len(random_scores)
print(f"Random Selection Score: {random_score:.4f}")

Random Selection Score: 570.6469


In [43]:
# Compare all influence scores
print("--- Influence Scores ---")
print(f"Greedy: {greedy_score:.4f}")
print(f"Stochastic Greedy: {stoch_greedy_score:.4f}")
print(f"Certain Failures: {cf_score:.4f}")
print(f"Observed Failures: {of_score:.4f}")
print(f"Random Selection: {random_score:.4f}")

--- Influence Scores ---
Greedy: 662.9355
Stochastic Greedy: 664.1450
Certain Failures: 500.3921
Observed Failures: 594.7595
Random Selection: 570.6469
